# Phase 2: Improved Models and Training

This notebook trains the Person 2 models for calorie prediction. The task is regression, not classification, so we optimize numeric calorie error and report MAE, MSE, RMSE, and R2.

Improvements in this version:

- Normalize calorie targets during training, then convert predictions back to real calories for evaluation
- Use SmoothL1 loss to reduce the effect of extreme calorie outliers
- Use a lighter CNN head with global average pooling to reduce overfitting
- Use AdamW, learning-rate scheduling, and early stopping
- Train MobileNetV2 in two phases: frozen backbone, then fine-tuning

## 1. Imports and Configuration

In [ ]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm.auto import tqdm
from sklearn.metrics import r2_score

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 16
CNN_EPOCHS = 25
TRANSFER_HEAD_EPOCHS = 10
TRANSFER_FINE_TUNE_EPOCHS = 15
WEIGHT_DECAY = 1e-4

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
IMAGE_DIR = DATA_DIR / 'images'
MODEL_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
MODEL_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

SPLIT_FILES = {
    'train': DATA_DIR / 'train.csv',
    'val': DATA_DIR / 'val.csv',
    'test': DATA_DIR / 'test.csv',
}

torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Project root: {PROJECT_ROOT}')
print(f'Using device: {device}')

## 2. Check Preprocessing Outputs

In [ ]:
required_paths = [IMAGE_DIR, SPLIT_FILES['train'], SPLIT_FILES['val'], SPLIT_FILES['test']]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing preprocessing outputs: ' + ', '.join(missing) + '. Run 01_data_preparation.ipynb or download_images.py first.')

print('All preprocessing outputs found.')

## 3. Target Scaling and Dataset

In [ ]:
class TargetScaler:
    def __init__(self, mean, std):
        self.mean = float(mean)
        self.std = float(std) if float(std) > 0 else 1.0

    @classmethod
    def from_series(cls, values):
        return cls(values.mean(), values.std())

    def transform(self, value):
        return (float(value) - self.mean) / self.std

    def inverse_array(self, values):
        return values * self.std + self.mean

    def save(self, path):
        path.write_text(json.dumps({'mean': self.mean, 'std': self.std}, indent=2), encoding='utf-8')


class NutritionDataset(Dataset):
    def __init__(self, csv_file, transform=None, target_scaler=None):
        self.data = pd.read_csv(csv_file)
        self.transform = transform
        self.target_scaler = target_scaler
        self.data['resolved_image_path'] = self.data['image_path'].apply(self.resolve_image_path)
        before = len(self.data)
        self.data = self.data[self.data['resolved_image_path'].apply(lambda path: path.exists())].reset_index(drop=True)
        skipped = before - len(self.data)
        if skipped:
            print(f'Skipped {skipped} rows from {Path(csv_file).name} because the image file was missing.')
        if self.data.empty:
            raise RuntimeError(f'No usable images found for {csv_file}.')

    @staticmethod
    def resolve_image_path(value):
        image_path = Path(value)
        if image_path.is_absolute():
            return image_path
        candidate = DATA_DIR / image_path
        if candidate.exists():
            return candidate
        return DATA_DIR / 'images' / image_path.name

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image_path = row['resolved_image_path']
        image = Image.open(image_path).convert('RGB')

        calories = float(row['calories'])
        if self.target_scaler:
            calories = self.target_scaler.transform(calories)
        calories = torch.tensor([calories], dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, calories

train_df = pd.read_csv(SPLIT_FILES['train'])
target_scaler = TargetScaler.from_series(train_df['calories'])
target_scaler.save(MODEL_DIR / 'target_scaler.json')
print(f'Target mean: {target_scaler.mean:.2f}, std: {target_scaler.std:.2f}')

## 4. DataLoaders

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_loader = DataLoader(NutritionDataset(SPLIT_FILES['train'], train_transforms, target_scaler), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(NutritionDataset(SPLIT_FILES['val'], eval_transforms, target_scaler), batch_size=BATCH_SIZE)
test_loader = DataLoader(NutritionDataset(SPLIT_FILES['test'], eval_transforms, target_scaler), batch_size=BATCH_SIZE)

print(f'Train images: {len(train_loader.dataset)}')
print(f'Validation images: {len(val_loader.dataset)}')
print(f'Test images: {len(test_loader.dataset)}')

## 5. CNN From Scratch

In [ ]:
class ScratchCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.35),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return self.regressor(x)

scratch_cnn = ScratchCNN().to(device)
print(scratch_cnn)

## 6. Transfer Learning: MobileNetV2

In [ ]:
def build_mobilenet_v2_regressor(freeze_backbone=True):
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
    for param in model.features.parameters():
        param.requires_grad = not freeze_backbone

    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 256),
        nn.ReLU(),
        nn.Dropout(p=0.2),
        nn.Linear(256, 1),
    )
    return model

mobilenet_v2 = build_mobilenet_v2_regressor(freeze_backbone=True).to(device)
print(mobilenet_v2.classifier)

## 7. Training Helpers

In [ ]:
def count_trainable_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)


def run_one_epoch(model, loader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()
    total_loss = 0.0
    context = torch.enable_grad() if is_training else torch.no_grad()

    with context:
        for images, targets in tqdm(loader, leave=False):
            images = images.to(device)
            targets = targets.to(device)

            if is_training:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, targets)

            if is_training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)

    return total_loss / len(loader.dataset)


def train_model(model, train_loader, val_loader, criterion, optimizer, epochs, save_path, patience=6):
    history = []
    best_val_loss = float('inf')
    epochs_without_improvement = 0
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        train_loss = run_one_epoch(model, train_loader, criterion, optimizer)
        val_loss = run_one_epoch(model, val_loader, criterion)
        scheduler.step(val_loss)

        history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
        print(f'Epoch {epoch:02d}/{epochs} | train loss: {train_loss:.4f} | val loss: {val_loss:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save(model.state_dict(), save_path)
            print(f'  Saved best weights: {save_path}')
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f'Early stopping after {epoch} epochs.')
                break

    return pd.DataFrame(history), time.time() - start_time


def predict(model, loader):
    model.eval()
    predictions = []
    targets = []
    with torch.no_grad():
        for images, y in tqdm(loader, leave=False):
            outputs = model(images.to(device)).cpu().numpy().reshape(-1)
            predictions.extend(outputs.tolist())
            targets.extend(y.numpy().reshape(-1).tolist())

    targets = target_scaler.inverse_array(np.array(targets))
    predictions = target_scaler.inverse_array(np.array(predictions))
    return targets, predictions


def evaluate_regression(model, loader):
    y_true, y_pred = predict(model, loader)
    mse = float(np.mean((y_true - y_pred) ** 2))
    return {
        'MAE': float(np.mean(np.abs(y_true - y_pred))),
        'MSE': mse,
        'RMSE': float(np.sqrt(mse)),
        'R2': float(r2_score(y_true, y_pred)),
    }

## 8. Train CNN From Scratch

In [ ]:
scratch_cnn = ScratchCNN().to(device)
criterion = nn.SmoothL1Loss(beta=0.5)
optimizer_cnn = optim.AdamW(scratch_cnn.parameters(), lr=8e-4, weight_decay=WEIGHT_DECAY)

print(f'Trainable parameters: {count_trainable_parameters(scratch_cnn):,}')
cnn_history, cnn_time = train_model(
    scratch_cnn,
    train_loader,
    val_loader,
    criterion,
    optimizer_cnn,
    epochs=CNN_EPOCHS,
    save_path=MODEL_DIR / 'cnn_from_scratch.pth',
)
cnn_history.to_csv(RESULTS_DIR / 'cnn_training_history.csv', index=False)

## 9. Train MobileNetV2 Head, Then Fine-Tune

In [ ]:
mobilenet_v2 = build_mobilenet_v2_regressor(freeze_backbone=True).to(device)
criterion = nn.SmoothL1Loss(beta=0.5)

print('Phase 1: train custom regression head with frozen backbone')
print(f'Trainable parameters: {count_trainable_parameters(mobilenet_v2):,}')
optimizer_head = optim.AdamW(filter(lambda p: p.requires_grad, mobilenet_v2.parameters()), lr=8e-4, weight_decay=WEIGHT_DECAY)
head_history, head_time = train_model(
    mobilenet_v2,
    train_loader,
    val_loader,
    criterion,
    optimizer_head,
    epochs=TRANSFER_HEAD_EPOCHS,
    save_path=MODEL_DIR / 'mobilenet_v2_head_best.pth',
)

mobilenet_v2.load_state_dict(torch.load(MODEL_DIR / 'mobilenet_v2_head_best.pth', map_location=device))
print('Phase 2: unfreeze final MobileNetV2 feature blocks for fine-tuning')
for param in mobilenet_v2.features[-7:].parameters():
    param.requires_grad = True

print(f'Trainable parameters after unfreezing: {count_trainable_parameters(mobilenet_v2):,}')
optimizer_fine = optim.AdamW(filter(lambda p: p.requires_grad, mobilenet_v2.parameters()), lr=2e-5, weight_decay=WEIGHT_DECAY)
fine_history, fine_time = train_model(
    mobilenet_v2,
    train_loader,
    val_loader,
    criterion,
    optimizer_fine,
    epochs=TRANSFER_FINE_TUNE_EPOCHS,
    save_path=MODEL_DIR / 'mobilenet_v2_finetuned.pth',
)

mobilenet_history = pd.concat([
    head_history.assign(phase='frozen_backbone'),
    fine_history.assign(phase='fine_tuning'),
], ignore_index=True)
mobilenet_history.to_csv(RESULTS_DIR / 'mobilenet_v2_training_history.csv', index=False)
mobilenet_time = head_time + fine_time

## 10. Test Evaluation and Model Comparison

In [ ]:
scratch_cnn.load_state_dict(torch.load(MODEL_DIR / 'cnn_from_scratch.pth', map_location=device))
mobilenet_v2.load_state_dict(torch.load(MODEL_DIR / 'mobilenet_v2_finetuned.pth', map_location=device))

comparison = pd.DataFrame([
    {
        'model': 'CNN from scratch',
        'trainable_parameters': count_trainable_parameters(scratch_cnn),
        'training_time_seconds': round(cnn_time, 2),
        **evaluate_regression(scratch_cnn, test_loader),
    },
    {
        'model': 'MobileNetV2 transfer learning',
        'trainable_parameters': count_trainable_parameters(mobilenet_v2),
        'training_time_seconds': round(mobilenet_time, 2),
        **evaluate_regression(mobilenet_v2, test_loader),
    },
])
comparison.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)
comparison

## 11. Save Full Models

In [ ]:
torch.save(scratch_cnn, MODEL_DIR / 'cnn_from_scratch_full.pt')
torch.save(mobilenet_v2, MODEL_DIR / 'mobilenet_v2_full.pt')
print('Saved final models in:', MODEL_DIR.resolve())